# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
- [Croissant JSON-LD Schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Let's load the dataset's metadata using `mlcroissant` and display the dataset name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

List record sets (`@id`), their available fields, and the IDs of each. This information will help identify how to reference data later.


In [ ]:
# List all record sets, their fields, and each field's @id
print("Available record sets and their fields:\n")
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets available in the Croissant metadata.")
else:
    for rs in record_sets:
        print(f"- Record Set: {rs.name}\n    @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                print(f"      - Field: {field.name} (@id: {field.id})")
        print()

## 3. Data Extraction

Let's extract one or more record sets and load them into Pandas DataFrames. We will reference record set and field `@id` values as recommended.


In [ ]:
# Identify available record sets
record_sets = metadata.record_sets
if not record_sets:
    raise ValueError("The dataset does not provide any record sets.")

# Let's extract all record_sets by their @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load each record set as a DataFrame
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    try:
        # Use generator to avoid loading large datasets in memory all at once
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(2))
    except Exception as e:
        print(f"  Could not load record set {rs_id}: {e}")
        dataframes[rs_id] = None

## 4. Exploratory Data Analysis (EDA)

Select a record set and a numeric field for demonstration. We'll filter, normalize, and group the data, referencing field `@id`s for all operations.

In [ ]:
# For demonstration, pick the first non-empty DataFrame and its fields

# Get a representative record set id with numeric columns
usable_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in record_sets:
    df = dataframes[rs.id]
    if df is not None and not df.empty:
        # Find a numeric field by Croissant type (Float/Number/Integer)
        for field in rs.fields:
            if hasattr(field, 'data_type') and field.data_type in ['Float', 'Integer', 'Number']:
                if field.id in df.columns:
                    usable_record_set_id = rs.id
                    numeric_field_id = field.id
                    break
        # Pick first string/categorical field for grouping, if exists
        for field in rs.fields:
            if hasattr(field, 'data_type') and field.data_type == 'Text' and field.id in df.columns:
                group_field_id = field.id
                break
    if usable_record_set_id and numeric_field_id:
        break

if usable_record_set_id is None or numeric_field_id is None:
    raise RuntimeError('No suitable numeric field found for analysis in any record set.')

df = dataframes[usable_record_set_id].copy()

# Convert numeric field to numeric dtype
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering (example: keep records above a threshold)
threshold = df[numeric_field_id].dropna().median()  # Use median as example threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records in '{usable_record_set_id}' with '{numeric_field_id}' > {threshold} (field @id):")
print(filtered_df.head())

# Normalization: z-score
col_norm = f"{numeric_field_id}_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nZ-score normalized '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, col_norm]].head())

# Optional grouping by a string/categorical field
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped by '{group_field_id}', mean of '{numeric_field_id}':")
    print(grouped.head())
else:
    print("\n(No appropriate categorical field found for grouping in this record set.)")

## 5. Visualization

Let's visualize the distribution of our chosen numeric field and, if possible, its relationship with the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if group_field_id exists)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    # Only take groups with enough samples for clarity
    value_counts = df[group_field_id].value_counts()
    top_groups = value_counts[value_counts>2].index[:6]  # up to 6 largest groups with >2 samples
    plot_df = df[df[group_field_id].isin(top_groups)]
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=plot_df)
    plt.title(f"'{numeric_field_id}' by '{group_field_id}' group")
    plt.show()

## 6. Conclusion

- **Record sets** and their **field `@id`s** were reviewed and referenced for all operations.
- **DataFrames** were created from each record set for further analysis.
- Example: We filtered and normalized a numeric field (`@id: {numeric_field_id}`) from record set `@id: {usable_record_set_id}`.
- Optionally, grouping and summary statistics by a text/categorical field (`@id: {group_field_id}`) were shown when available.
- Distribution and boxplot visualizations provide initial insight into the feature distributions.

For advanced analyses, reference field and record set IDs (as shown) to ensure reproducibility and alignment with the Croissant metadata. See the [FAIR^2 documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for more domain details.